# CISB5123 Text Analytics – Lab Assignment 3 (Topic Modeling)

# Name:Bawazir Ahmed Ali Ahmed
# Student ID: SW01084040   

# Name:ALSAKKAF MUSAAB AKRAM
# Student ID: SW01083207

In [1]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from gensim import corpora
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel

In [2]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ahmed\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ahmed\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ahmed\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [4]:
df = pd.read_csv(r'C:\Users\ahmed\Desktop\Text Analytics\Lab Assignment 3\news_dataset.csv')

df = df[['text']]
df.dropna(inplace=True)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (11096, 1)


,text
0,I was wondering if anyone out there could enli...
1,I recently posted an article asking what kind ...
2,\nIt depends on your priorities. A lot of peo...
3,an excellent automatic can be found in the sub...
4,: Ford and his automobile. I need information...


In [5]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    words = text.split()
    
    words = [w for w in words if w not in stop_words]
    words = [stemmer.stem(w) for w in words]
    words = [lemmatizer.lemmatize(w) for w in words]
    
    return words

processed_text = df['text'].apply(clean_text)

processed_text.head()

0    [wonder, anyon, could, enlighten, car, saw, da...
1    [recent, post, articl, ask, kind, rate, singl,...
2    [depend, prioriti, lot, peopl, put, higher, pr...
3    [excel, automat, found, subaru, legaci, switch...
4    [ford, automobil, need, inform, whether, ford,...
Name: text, dtype: object

In [6]:
dictionary = corpora.Dictionary(processed_text)

dictionary.filter_extremes(no_below=5, no_above=0.5)

corpus = [dictionary.doc2bow(text) for text in processed_text]

print("Sample:", corpus[0])

Sample: [(0, 1), (1, 2), (2, 1), (3, 1), (4, 1), (5, 4), (6, 1), (7, 1), (8, 2), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 1), (16, 1), (17, 1), (18, 1), (19, 2), (20, 1), (21, 1), (22, 1), (23, 1), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (30, 1), (31, 1), (32, 1), (33, 1), (34, 1)]


In [7]:
lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=4,
    random_state=42,
    passes=15
)

In [11]:
topics = lda_model.print_topics(num_topics=4, num_words=5)

for i, topic in topics:
    print(f"Topic {i+1}: {topic}")

Topic 1: 0.014*"maxaxaxaxaxaxaxaxaxaxaxaxaxaxax" + 0.014*"use" + 0.008*"get" + 0.008*"one" + 0.008*"window"
Topic 2: 0.022*"x" + 0.015*"use" + 0.014*"key" + 0.009*"encrypt" + 0.008*"file"
Topic 3: 0.009*"year" + 0.009*"game" + 0.007*"team" + 0.006*"play" + 0.005*"new"
Topic 4: 0.009*"would" + 0.009*"peopl" + 0.009*"one" + 0.006*"think" + 0.006*"dont"


In [9]:
coherence_model = CoherenceModel(
    model=lda_model,
    texts=processed_text,
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()
print("Coherence Score:", coherence_score)

Coherence Score: 0.5451788371718148


In [10]:
print("""
The coherence score measures the degree of semantic similarity between words in each topic.
A higher coherence score indicates that the topics generated by the LDA model are more meaningful
and interpretable.

In this model, the coherence score suggests that the topics are reasonably coherent, meaning that
the model has successfully grouped related words together into distinct topics. However, further
improvements could be made by adjusting parameters such as the number of topics or preprocessing steps.
""")


The coherence score measures the degree of semantic similarity between words in each topic.
A higher coherence score indicates that the topics generated by the LDA model are more meaningful
and interpretable.

In this model, the coherence score suggests that the topics are reasonably coherent, meaning that
the model has successfully grouped related words together into distinct topics. However, further
improvements could be made by adjusting parameters such as the number of topics or preprocessing steps.

